# Run Saved Gesture Model (Webcam)

This notebook loads the exported DollarPy template model from `models/gesture_recognizer_dollarpy.pth` and opens webcam classification.

- Run cells from top to bottom.
- Press **q** in the camera window to quit.

In [1]:
%pip install opencv-python mediapipe dollarpy pickle

  Using cached opencv_python-4.13.0.92-cp37-abi3-win_amd64.whl.metadata (20 kB)
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement pickle (from versions: none)
ERROR: No matching distribution found for pickle


In [2]:
%pip install torch 

Note: you may need to restart the kernel to use updated packages.


In [42]:
import os
os.environ["GLOG_minloglevel"] = "3"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import cv2
import mediapipe as mp
from mediapipe.python.solutions import drawing_utils, hands
from dollarpy import Point, Recognizer
from pathlib import Path
from typing import List, Tuple
from collections import deque
import torch
import pickle

mp_drawing = drawing_utils
mp_hands = hands

MODEL_PATH = Path("models/gesture_recognizer_dollarpy.pth")
CAMERA_INDEX = 0
LEFT_HAND_STROKE_ID = 1
RIGHT_HAND_STROKE_ID = 101
TRACK_LANDMARK_ID = 8

In [43]:
def extract_hand_points(hand_results) -> List[Point]:
    frame_points: List[Point] = []

    if not hand_results.multi_hand_landmarks:
        return frame_points

    for hand_idx, hand_landmarks in enumerate(hand_results.multi_hand_landmarks):
        tip = hand_landmarks.landmark[TRACK_LANDMARK_ID]

        is_left_hand = False
        if hand_results.multi_handedness and hand_idx < len(hand_results.multi_handedness):
            label = hand_results.multi_handedness[hand_idx].classification[0].label.lower()
            is_left_hand = label == "left"

        stroke_id = LEFT_HAND_STROKE_ID if is_left_hand else RIGHT_HAND_STROKE_ID
        frame_points.append(Point(tip.x, tip.y, stroke_id))

    return frame_points


def classify_points(recognizer: Recognizer, points: List[Point], threshold: float) -> Tuple[str, float]:
    try:
        prediction = recognizer.recognize(points)
    except ZeroDivisionError:
        return "unknown", 0.0

    label = prediction[0]
    score = prediction[1] if len(prediction) > 1 else 0.0
    if score < threshold:
        return "unknown", score
    return label, score


def run_realtime_classification(recognizer: Recognizer, window_size: int, camera_index: int, threshold: float, min_det: float, min_track: float):
    cap = cv2.VideoCapture(camera_index)
    if not cap.isOpened():
        raise ValueError(f"Could not open camera index: {camera_index}")

    point_buffer = deque(maxlen=window_size)

    with mp_hands.Hands(
        static_image_mode=False,
        max_num_hands=2,
        model_complexity=1,
        min_detection_confidence=min_det,
        min_tracking_confidence=min_track,
    ) as hand_tracker:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            frame = cv2.flip(frame, 1)
            image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image_rgb.flags.writeable = False
            hand_results = hand_tracker.process(image_rgb)
            image_rgb.flags.writeable = True
            image_bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)

            frame_points = extract_hand_points(hand_results)
            point_buffer.extend(frame_points)

            prediction_text = "Collecting frames..."
            if len(point_buffer) >= point_buffer.maxlen:
                pred_label, pred_score = classify_points(recognizer, list(point_buffer), threshold)
                prediction_text = f"{pred_label} ({pred_score:.3f})"

            if hand_results.multi_hand_landmarks:
                for hand_landmarks in hand_results.multi_hand_landmarks:
                    mp_drawing.draw_landmarks(image_bgr, hand_landmarks, mp_hands.HAND_CONNECTIONS)

            cv2.putText(
                image_bgr,
                prediction_text,
                (20, 30),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0, 255, 0),
                2,
                cv2.LINE_AA,
            )
            cv2.imshow("Realtime Classification", image_bgr)

            if cv2.waitKey(1) & 0xFF == ord("q"):
                break

    cap.release()
    cv2.destroyAllWindows()
    cv2.waitKey(50)

In [44]:


def load_payload(model_path: Path):
    if not model_path.exists():
        raise FileNotFoundError(f"Model file not found: {model_path.resolve()}")

    torch_error = None
    try:
        try:
            payload = torch.load(model_path, map_location="cpu", weights_only=False)
        except TypeError:
            payload = torch.load(model_path, map_location="cpu")
        return payload, "torch.load"
    except Exception as exc:
        torch_error = exc

    try:
        with model_path.open("rb") as f:
            payload = pickle.load(f)
        return payload, "pickle"
    except Exception as exc:
        raise RuntimeError(
            f"Failed to load model with torch and pickle. Torch error: {torch_error}. Pickle error: {exc}"
        )


payload, loader = load_payload(MODEL_PATH)
recognizer = Recognizer(payload["templates"])

cfg = payload.get("config", {})
CLASSIFICATION_THRESHOLD = float(cfg.get("classification_threshold", 0.05))
WINDOW_SIZE = int(cfg.get("window_size", 45))
MIN_DETECTION_CONFIDENCE = float(cfg.get("min_detection_confidence", 0.5))
MIN_TRACKING_CONFIDENCE = float(cfg.get("min_tracking_confidence", 0.5))

print(f"Loaded model: {MODEL_PATH.resolve()}")
print(f"Loader: {loader}")
print(f"Classes: {len(payload.get('class_names', []))}")
print(f"Window size: {WINDOW_SIZE} | Threshold: {CLASSIFICATION_THRESHOLD}")

Loaded model: C:\mine\HCI\HCI Project\TUIO11_NET-master\models\gesture_recognizer_dollarpy.pth
Loader: torch.load
Classes: 10
Window size: 45 | Threshold: 0.05


In [45]:
# Run webcam inference. Press q to close the window.
run_realtime_classification(
    recognizer=recognizer,
    window_size=WINDOW_SIZE,
    camera_index=CAMERA_INDEX,
    threshold=CLASSIFICATION_THRESHOLD,
    min_det=MIN_DETECTION_CONFIDENCE,
    min_track=MIN_TRACKING_CONFIDENCE,
)